# A silenced role grants no access.

Run the cells in order. The matching [script](16_silenced_role.py) has the same declarations, calls and assertions.

See the [guide](../../docs/how-to/roles-and-object-rules.md).

## Declarations

Declare the domain, role requirements and typed operation before asking the machine.

In [1]:
"""A silenced role grants no access.
Run: .venv/bin/python examples/step_03_authorization_and_roles/16_silenced_role.py
"""

from __future__ import annotations

import asyncio

from pydantic import Field

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.context import Context
from aoa.action_machine.context.user_info import UserInfo
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.intents.role_mode import RoleMode, role_mode
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.model.base_state import BaseState
from aoa.action_machine.resources import Connections
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox

# %% Setup


class StoreDomain(BaseDomain):
    """Group the order operations."""

    name = "store"
    description = "Order management"


class ManagerRole(ApplicationRole):
    """Permit order management."""

    name = "manager"
    description = "Can manage orders"


class OrderParams(BaseParams):
    """Identify the order to cancel."""

    order_id: str = Field(description="Order identifier")


class OrderResult(BaseResult):
    """Report the cancelled order."""

    order_id: str = Field(description="Cancelled order identifier")


@role_mode(RoleMode.SILENCED)
class TemporaryManagerRole(ManagerRole):
    """Suspend temporary manager permissions."""

    name = "temporary_manager"
    description = "Temporary manager access"


@meta(description="Cancel an order", domain=StoreDomain)
@check_roles(ManagerRole)
class CancelOrderAction(BaseAction[OrderParams, OrderResult]):
    """Cancel an order after checking access."""

    @summary_aspect("Cancel the order")
    async def cancel_summary(
        self, params: OrderParams, state: BaseState, box: ToolsBox, connections: Connections
    ) -> OrderResult:
        """Return the cancellation result."""
        return OrderResult(order_id=params.order_id)


## Step 1

An active manager passes. A silenced descendant does not satisfy the manager requirement.

In [2]:
machine = ActionProductMachine(cache_coordinator=None)
active = Context(user=UserInfo(user_id="m-1", roles=(ManagerRole,)))
suspended = Context(user=UserInfo(user_id="t-1", roles=(TemporaryManagerRole,)))
params = OrderParams(order_id="ord-001")


## Step 2

An active manager passes. A silenced descendant does not satisfy the manager requirement.

In [3]:
answer = await machine.check_access_decide(active, CancelOrderAction, params)
assert answer.model_dump(mode="json") == {"kind": "allowed"}
print("active:", answer.model_dump(mode="json"))
answer = await machine.check_access_decide(suspended, CancelOrderAction, params)
assert answer.model_dump(mode="json") == {"kind": "refused", "gate": "CHECK_ROLES", "reason": None}
print("silenced:", answer.model_dump(mode="json"))


active: {'kind': 'allowed'}
silenced: {'kind': 'refused', 'gate': 'CHECK_ROLES', 'reason': None}
